In [1]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
!pip install youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [ ]:
%pip install -U youtube-transcript-api langchain langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [1]:
%pip install --upgrade jupyter ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   --------------------- ---------------- 524.3/914.9 kB 410.7 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 410.7 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 410.7 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 410.7 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/914.9 kB 410.7 kB/s eta 0:00:01
   -------------------------------- ----- 786.4/914.9 kB 341.4 kB/s eta 0:00:01
   -------------------------------------- 914.9/914.9 kB 368.6 kB/s eta 0:00:00
   -----


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled




Step 1a - Indexing (Document Ingestion)

In [ ]:
!pip show youtube-transcript-api

In [2]:
video_id = "Gfr50f6ZBvo"

try:
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id)

    transcript = " ".join([t.text for t in transcript_list])
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

except Exception as e:
    print("Error:", str(e))

NameError: name 'TranscriptsDisabled' is not defined

Step 1b - Indexing (Text Splitting)

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [16]:
len(chunks)

168

In [17]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
print(vector_store.index_to_docstore_id)

{0: '42358df5-5302-45a6-8940-8b8c50eb9206', 1: 'd6716c3b-1024-478e-b96c-affc80cd0f8e', 2: 'fbefd100-c30a-4402-8d47-9a22d1fbfccb', 3: '5f4f5be0-b2fe-4c3e-b3d9-5b9ed25f3a4d', 4: 'c166a199-92db-4d9d-a56a-8b75362ae570', 5: 'c85a69df-ac5c-4ead-973c-d86a8d136db9', 6: '5fbf8780-d5ae-446b-ac5e-8482463cd2a5', 7: '16fa51e6-d31c-4787-a0ef-c4098aec0e9e', 8: 'd28e1c80-a62f-4db9-b536-1388936f8515', 9: '0aa5433c-3531-429d-a0f7-10a4df1a89d8', 10: 'ed07fbb5-ec2c-472d-8993-d0911bc52494', 11: 'f00f2e21-eafe-43e4-92c4-8b9d72f239a0', 12: '6ecdcfec-b73e-4672-ad27-0b847980fa00', 13: '915f0972-ddca-4e9c-a06a-3c2bb7501a1b', 14: 'ab9dfc1e-9b55-46b7-98ff-006311b438d4', 15: 'a8e83f68-4d76-4ebe-b266-f421a990d07f', 16: '5b600d9e-69d6-4285-bf44-23d032902506', 17: '825ccc71-63d8-4414-805e-168d9f92cbef', 18: '595bde96-e6c3-402b-a11c-bca6b6d42be9', 19: '76ce7a88-6ad2-4798-932b-e479e3aca381', 20: '3890858b-bddc-40ee-8db7-40c194f8690e', 21: '985b9b22-36ee-47c6-b3f6-d079fb28c858', 22: 'f6358689-587d-418f-a4df-f17dda39d3f8

Step 2 - Retrieval

In [23]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [26]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001AC3127BE10>, search_kwargs={'k': 4})

In [ ]:
retriever.invoke('What is deepmind')

Step 3 - Augmentation

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [29]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question) 

In [ ]:
retrieved_docs 

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')